# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

### **EQUIPE**

---

**Integrante 01:**

`Gustavo Oliveira Rafael - 11202230857`

**Integrante 02:**

`Henrique Simões Bonfim - `

**Integrante 03:**

`Lívia de Pádua Ferné - `

**Integrante 04:**

`Henrique Ishida - `

### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

O modelo foi escolhido por ser otimizado para extração de informações, processamento de documentos e fluxos com agentes, priorizando baixa latência e custo. Também suporta saídas estruturadas, importantes para gerar casos de teste em JSON. A integração com LangChain pode ser realizada pela biblioteca langchain-google-genai, utilizando ChatGoogleGenerativeAI.

>


**LLM**: Google Gemini 3.5 Flash-Lite (gemini-3.5-flash-lite)

>

**Link para a documentação oficial**: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite?utm_source=chatgpt.com



### **API (Opcional)**
---

**API**: Swagger Petstore API — OpenAPI 3.0

**Site oficial**: https://petstore3.swagger.io/

**Link para a documentação oficial**: https://learn.openapis.org/examples/v3.0/petstore.html






### **IMPLEMENTAÇÃO**
---

## **1. Visão geral do projeto**

**Gerador Inteligente de Testes a partir de Documentação de API**

O projeto transforma a documentação de uma API pública (especificação **OpenAPI** da Swagger Petstore) em **casos de teste executáveis**. Um LLM lê as descrições dos endpoints, gera cenários de teste e um *harness* em Python executa requisições HTTP reais e **julga** cada caso, sem depender do LLM.

**Questão central:** até que ponto um LLM consegue converter documentação de API em testes úteis, mantendo a avaliação objetiva e reproduzível?

**Princípio de projeto:** *o agente planeja e propõe; o harness executa e julga.* "Aprovado" ou "reprovado" nunca é uma opinião do modelo.

**Pipeline**

| Etapa | O que acontece | Quem faz |
|---|---|---|
| 1. Coleta | Baixa e valida o OpenAPI; seleciona as operações permitidas | Python |
| 2. RAG | Indexa trechos da documentação e recupera os pertinentes a cada endpoint | LangChain + TF-IDF |
| 3. PLN | **Classificação de Textos** e **Reconhecimento de Entidades Nomeadas** | LangChain + Gemini |
| 4. Geração | Casos de teste estruturados com dados sintéticos válidos e inválidos | LangChain + Gemini |
| 5. Execução | Requisições HTTP com timeout, limites e regras de segurança | Python (harness) |
| 6. Julgamento | Validadores determinísticos comparam resposta × expectativa | Python (harness) |
| Extra | Agente que consulta a documentação e solicita testes via ferramentas controladas | LangChain + Gemini |

**Técnicas de PLN (lista oficial do enunciado):** Classificação de Textos e Reconhecimento de Entidades Nomeadas, ambas aplicadas pelo LLM sobre o **texto** das descrições. **Extras explorados:** RAG, geração de dados sintéticos e agentes.

## **2. Configuração do ambiente**

A chave da API do Gemini deve estar cadastrada em **Secrets** do Colab (ícone de chave, no menu à esquerda) com o nome `GOOGLE_API_KEY` e o acesso ao notebook habilitado. A chave nunca é impressa nem salva em arquivo.

In [ ]:
# Instalação das dependências. Se o Colab pedir para reiniciar o ambiente, reinicie e execute o notebook novamente.
!pip install -q -U langchain langchain-core langchain-google-genai pydantic jsonschema scikit-learn requests pandas matplotlib

In [ ]:
import os, re, json, time, getpass, shutil
from pathlib import Path
from urllib.parse import urljoin, urlparse, quote
from typing import List, Literal, Optional, Any

import requests
import pandas as pd
import matplotlib.pyplot as plt
import jsonschema
from pydantic import BaseModel, Field, PrivateAttr

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)

# ----------------------------- Parâmetros do projeto -----------------------------
MODEL_ID = "gemini-3.5-flash-lite"
OPENAPI_URL = "https://petstore3.swagger.io/api/v3/openapi.json"

# Escopo inicial: somente leituras (GET). Nada destrutivo, nada que altere dados compartilhados.
OPERACOES_PERMITIDAS = [
    ("GET", "/pet/findByStatus"),
    ("GET", "/pet/{petId}"),
    ("GET", "/store/inventory"),
]

HTTP_TIMEOUT = 15            # segundos por requisição
HTTP_TENTATIVAS_EXTRAS = 2   # re-tentativas para falhas transitórias (rede, 429, 5xx)
MAX_BYTES_RESPOSTA = 2_000_000
MAX_CASOS_POR_OPERACAO = 8
PAUSA_ENTRE_CHAMADAS = 1.0   # segundos entre chamadas ao LLM (respeita cotas)
PETSTORE_API_KEY = "special-key"  # chave pública de demonstração da Petstore (só para operações com api_key)
USAR_CACHE_LLM = False       # True reaproveita os JSONs de results/ e evita novas chamadas ao LLM

RESULTADOS = Path("results")
RESULTADOS.mkdir(exist_ok=True)

# ----------------------------- Chave da API do Gemini -----------------------------
def obter_chave_google() -> str:
    """Lê a chave dos Secrets do Colab; se não estiver no Colab, usa variável de ambiente ou pede ao usuário."""
    try:
        from google.colab import userdata
        chave = userdata.get("GOOGLE_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    chave = os.environ.get("GOOGLE_API_KEY")
    if chave:
        return chave
    return getpass.getpass("Informe a GOOGLE_API_KEY: ")

os.environ["GOOGLE_API_KEY"] = obter_chave_google()

def texto_da_resposta(msg) -> str:
    """Extrai texto de uma mensagem do LLM (o conteúdo pode vir como string ou lista de blocos)."""
    conteudo = getattr(msg, "content", msg)
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return "".join(b if isinstance(b, str) else b.get("text", "") for b in conteudo if isinstance(b, (str, dict)))
    return str(conteudo)

# ----------------------------- LLM via LangChain -----------------------------
# Obs.: para modelos Gemini 3.x o Google recomenda manter a temperatura padrão. A reprodutibilidade
# do projeto vem do harness determinístico e dos JSONs salvos em results/, não da temperatura.
llm = ChatGoogleGenerativeAI(model=MODEL_ID, max_retries=2, timeout=90)

In [ ]:
# Teste de sanidade: o LLM responde? (valida chave, identificador do modelo e cota)
_resp = llm.invoke("Responda apenas com a palavra: OK")
print(f"Modelo: {MODEL_ID}")
print("Resposta do LLM:", texto_da_resposta(_resp).strip())

## **3. Coleta e tratamento do corpus (OpenAPI)**

Baixamos o arquivo OpenAPI, validamos a estrutura mínima e **normalizamos** cada operação permitida em um registro que mantém a **origem** de cada informação (caminho dentro do documento). As referências `$ref` são resolvidas para que os *schemas* de resposta possam ser usados pelo harness.

In [ ]:
def carregar_openapi():
    """Baixa o OpenAPI; se a rede falhar e houver cópia local em results/, usa a cópia."""
    cache = RESULTADOS / "openapi_petstore.json"
    try:
        r = requests.get(OPENAPI_URL, timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        spec = r.json()
        cache.write_text(json.dumps(spec, ensure_ascii=False, indent=2), encoding="utf-8")
        return spec, "rede"
    except Exception as e:
        if cache.exists():
            print(f"Aviso: falha ao baixar ({e!r}); usando cópia local.")
            return json.loads(cache.read_text(encoding="utf-8")), "cópia local"
        raise

SPEC, ORIGEM_SPEC = carregar_openapi()

# Validação estrutural mínima do documento
assert str(SPEC.get("openapi", "")).startswith("3."), "O documento não é OpenAPI 3.x"
assert "paths" in SPEC and "components" in SPEC, "Faltam 'paths' ou 'components'"
for _m, _p in OPERACOES_PERMITIDAS:
    assert _p in SPEC["paths"] and _m.lower() in SPEC["paths"][_p], f"Operação ausente: {_m} {_p}"

# O campo servers[0].url é relativo ("/api/v3"); a URL base é resolvida a partir do endereço do documento.
BASE_URL = urljoin(OPENAPI_URL, SPEC["servers"][0]["url"]).rstrip("/")
HOST_PERMITIDO = urlparse(BASE_URL).netloc

print(f"Origem: {ORIGEM_SPEC} | OpenAPI {SPEC['openapi']} | {SPEC['info']['title']} v{SPEC['info']['version']}")
print(f"Caminhos no documento: {len(SPEC['paths'])} | Selecionados: {len(OPERACOES_PERMITIDAS)}")
print(f"URL base: {BASE_URL}")

In [ ]:
def resolver_ref(spec, ref):
    """Segue uma referência local do tipo '#/components/schemas/Pet'."""
    no = spec
    for parte in ref.lstrip("#/").split("/"):
        no = no[parte]
    return no

def expandir_refs(spec, no, profundidade=0):
    """Substitui todos os $ref pelo conteúdo referenciado e remove campos irrelevantes para validação."""
    if profundidade > 10:
        return {}
    if isinstance(no, dict):
        if "$ref" in no:
            return expandir_refs(spec, resolver_ref(spec, no["$ref"]), profundidade + 1)
        saida = {k: expandir_refs(spec, v, profundidade + 1) for k, v in no.items() if k not in ("xml", "example", "externalDocs")}
        if saida.get("nullable") is True and isinstance(saida.get("type"), str):  # OpenAPI 3.0 -> JSON Schema
            saida["type"] = [saida["type"], "null"]
        return saida
    if isinstance(no, list):
        return [expandir_refs(spec, x, profundidade + 1) for x in no]
    return no

def construir_registro(spec, metodo, caminho):
    """Registro normalizado de uma operação, preservando textos originais e a origem de cada parte."""
    op = spec["paths"][caminho][metodo.lower()]
    parametros = []
    for p in op.get("parameters", []):
        p = expandir_refs(spec, p)
        parametros.append({"nome": p["name"], "local": p["in"], "obrigatorio": bool(p.get("required", False)),
                           "descricao": p.get("description", ""), "schema": p.get("schema", {})})
    respostas = {}
    for codigo, r in op["responses"].items():
        r = expandir_refs(spec, r)
        respostas[codigo] = {"descricao": r.get("description", ""),
                             "schema_json": r.get("content", {}).get("application/json", {}).get("schema")}
    return {
        "id": f"{metodo} {caminho}", "metodo": metodo, "caminho": caminho, "operationId": op.get("operationId", ""),
        "resumo": op.get("summary", ""), "descricao": op.get("description", ""), "tags": op.get("tags", []),
        "parametros": parametros, "respostas": respostas,
        "seguranca": sorted({nome for req in op.get("security", []) for nome in req}),
        "schemas_referenciados": sorted(set(re.findall(r"#/components/schemas/(\w+)", json.dumps(op)))),
        "origem": f"paths.{caminho}.{metodo.lower()}",
    }

REGISTROS = {f"{m} {p}": construir_registro(SPEC, m, p) for m, p in OPERACOES_PERMITIDAS}

pd.DataFrame([{
    "operação": r["id"], "resumo": r["resumo"],
    "parâmetros": ", ".join(f"{p['nome']} ({p['local']})" for p in r["parametros"]) or "—",
    "códigos documentados": ", ".join(r["respostas"].keys()),
} for r in REGISTROS.values()])

## **4. RAG: indexação e recuperação da documentação**

Cada operação é quebrada em **trechos** (resumo/descrição, cada parâmetro, cada resposta, segurança e *schemas* de componentes). Cada trecho recebe um `chunk_id` e a sua origem no documento OpenAPI. Antes de cada chamada ao LLM, os trechos pertinentes ao endpoint são **recuperados** por similaridade TF-IDF, usando um `BaseRetriever` do LangChain. O LLM é obrigado a **citar** os `chunk_id` que sustentam cada resposta, o que dá rastreabilidade entre documentação, caso gerado e evidência de execução.

In [ ]:
def compactar(obj, limite=1500):
    s = json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    return s if len(s) <= limite else s[:limite] + "…"

def descrever_schema_componente(nome, schema):
    props = []
    for pn, pv in schema.get("properties", {}).items():
        if "$ref" in pv:
            tipo = f"object ({pv['$ref'].split('/')[-1]})"
        elif pv.get("type") == "array":
            item = pv.get("items", {})
            tipo = f"array of {item['$ref'].split('/')[-1] if '$ref' in item else item.get('type', 'any')}"
        else:
            tipo = pv.get("type", "any") + (f", {pv['format']}" if "format" in pv else "")
            if "enum" in pv:
                tipo += f", enum: {pv['enum']}"
        props.append(f"{pn} ({tipo})")
    obrig = schema.get("required", [])
    return f"Schema {nome}: required fields: {obrig if obrig else 'none'}; properties: " + "; ".join(props) + "."

def construir_documentos(spec, registros):
    docs, schemas_por_op = [], {}

    def adicionar(op_id, tipo, sufixo, texto, origem):
        docs.append(Document(page_content=texto, metadata={
            "chunk_id": f"{op_id}#{sufixo}", "endpoints": op_id, "tipo": tipo, "origem": origem}))

    for op_id, rec in registros.items():
        for nome in rec["schemas_referenciados"]:
            schemas_por_op.setdefault(nome, []).append(op_id)
        adicionar(op_id, "resumo", "resumo",
                  f"Operation summary and description: {rec['resumo']} {rec['descricao']}".strip(), rec["origem"])
        for p in rec["parametros"]:
            adicionar(op_id, "parametro", f"param-{p['nome']}",
                      f"Parameter '{p['nome']}' (location: {p['local']}, required: {str(p['obrigatorio']).lower()}): "
                      f"{p['descricao']} Schema: {compactar(p['schema'])}", f"{rec['origem']}.parameters")
        for codigo, r in rec["respostas"].items():
            txt = f"Response {codigo}: {r['descricao']}."
            if r["schema_json"]:
                txt += f" JSON body schema: {compactar(r['schema_json'])}"
            adicionar(op_id, "resposta", f"resp-{codigo}", txt, f"{rec['origem']}.responses.{codigo}")
        if rec["seguranca"]:
            adicionar(op_id, "seguranca", "seguranca",
                      f"Security schemes accepted: {', '.join(rec['seguranca'])}.", f"{rec['origem']}.security")
    # Trechos de schemas de componentes (compartilhados entre operações)
    for nome, ops in schemas_por_op.items():
        docs.append(Document(page_content=descrever_schema_componente(nome, spec["components"]["schemas"][nome]),
                             metadata={"chunk_id": f"schema:{nome}", "endpoints": "|".join(ops), "tipo": "schema",
                                       "origem": f"components.schemas.{nome}"}))
    return docs

DOCUMENTOS = construir_documentos(SPEC, REGISTROS)

class RetrieverTfidf(BaseRetriever):
    """Retriever local (sem custo de API): TF-IDF + similaridade de cosseno, com filtro por endpoint."""
    documentos: List[Document]
    k: int = 6
    endpoint: Optional[str] = None
    _vec: Any = PrivateAttr(default=None)
    _matriz: Any = PrivateAttr(default=None)

    def model_post_init(self, __context):
        self._vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), sublinear_tf=True)
        self._matriz = self._vec.fit_transform([d.page_content for d in self.documentos])

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        sims = cosine_similarity(self._vec.transform([query]), self._matriz)[0]
        saida = []
        for i in sorted(range(len(sims)), key=lambda j: sims[j], reverse=True):
            d = self.documentos[i]
            if self.endpoint and self.endpoint not in d.metadata["endpoints"].split("|"):
                continue
            saida.append(Document(page_content=d.page_content, metadata={**d.metadata, "score": round(float(sims[i]), 4)}))
            if len(saida) >= self.k:
                break
        return saida

RETRIEVER_BASE = RetrieverTfidf(documentos=DOCUMENTOS)

def recuperar(op_id: str, consulta: str, k: int = 6) -> List[Document]:
    return RETRIEVER_BASE.model_copy(update={"endpoint": op_id, "k": k}).invoke(consulta)

def formatar_contexto(docs: List[Document]) -> str:
    return "\n\n".join(f"[{d.metadata['chunk_id']}] {d.page_content}" for d in docs)

def consulta_geral(op_id: str) -> str:
    r = REGISTROS[op_id]
    return f"{r['resumo']} {r['descricao']} parameters responses status codes schema"

print(f"{len(DOCUMENTOS)} trechos indexados.\n")
_docs_demo = recuperar("GET /pet/{petId}", "What happens when the pet does not exist?", k=4)
pd.DataFrame([{"chunk_id": d.metadata["chunk_id"], "tipo": d.metadata["tipo"], "score": d.metadata["score"],
               "origem": d.metadata["origem"]} for d in _docs_demo])

## **5. Técnicas de PLN com LangChain + Gemini**

Duas tarefas **separadas**, cada uma com seu prompt e seu *schema* de saída (Pydantic). A saída é validada pelo *schema* e depois por checagens determinísticas:

1. **Classificação de Textos:** intenção (consulta, criação, atualização, exclusão), risco e natureza do endpoint, a partir do **texto** da descrição. O verbo HTTP **não** é informado ao modelo; depois comparamos a resposta com o verbo real.
2. **Reconhecimento de Entidades Nomeadas (NER):** recursos, identificadores, parâmetros, campos, restrições e códigos de resposta, cada um com o **trecho de origem**. Verificamos se o trecho citado existe literalmente na documentação, para detectar entidades inventadas.

In [ ]:
# ----------------------------- Schemas de saída (Pydantic) -----------------------------
class ClassificacaoOperacao(BaseModel):
    """Classificação textual de uma operação de API."""
    intencao: Literal["consulta", "criacao", "atualizacao", "exclusao"] = Field(description="Intenção principal da operação, inferida do texto.")
    risco: Literal["baixo", "medio", "alto"] = Field(description="Risco de efeito colateral: baixo = apenas leitura; alto = altera ou apaga dados.")
    natureza: Literal["leitura_de_recurso_unico", "listagem_filtrada", "agregacao_estatistica", "escrita", "outro"] = Field(description="Natureza do endpoint.")
    justificativa: str = Field(description="Justificativa curta (no máximo 2 frases, em português).")
    evidencias: List[str] = Field(description="Lista de chunk_id (entre colchetes no contexto) que sustentam a classificação.")

class Entidade(BaseModel):
    tipo: Literal["recurso", "identificador", "parametro", "campo", "restricao", "codigo_resposta", "valor_enumerado"] = Field(description="Tipo da entidade.")
    valor: str = Field(description="Valor da entidade, exatamente como aparece no texto (ex.: petId, 404, available).")
    trecho_origem: str = Field(description="Cópia LITERAL, caractere a caractere, do trecho do contexto onde a entidade aparece.")

class ExtracaoEntidades(BaseModel):
    entidades: List[Entidade]

class Parametro(BaseModel):
    nome: str = Field(description="Nome do parâmetro, exatamente como documentado.")
    local: Literal["path", "query"] = Field(description="Onde o parâmetro vai na requisição.")
    valor: str = Field(description="Valor sintético, sempre como string.")

class Verificacao(BaseModel):
    """Verificação de conteúdo com vocabulário FECHADO, executada pelo harness (não pelo LLM)."""
    tipo: Literal["json_raiz_tipo", "json_tem_chaves", "todos_itens_campo_em", "todos_valores_inteiros", "campo_igual"] = Field(
        description="json_raiz_tipo: valores=['array'] ou ['object']; json_tem_chaves: valores=chaves obrigatórias do objeto raiz; "
                    "todos_itens_campo_em: raiz é array e item[campo] deve estar em valores; todos_valores_inteiros: raiz é objeto e todos os valores são inteiros; "
                    "campo_igual: raiz é objeto e str(raiz[campo]) == valores[0].")
    campo: Optional[str] = Field(description="Campo usado pela verificação, ou null quando não se aplica.")
    valores: List[str] = Field(description="Valores de referência da verificação (strings).")

class CasoTeste(BaseModel):
    nome: str = Field(description="Nome curto do caso.")
    objetivo: str = Field(description="O que o caso verifica, em uma frase.")
    tipo: Literal["positivo", "negativo"]
    parametros: List[Parametro] = Field(description="Parâmetros da requisição (lista vazia se a operação não tem parâmetros).")
    codigos_esperados: List[int] = Field(description="Códigos HTTP aceitáveis, SOMENTE entre os documentados.")
    verificacoes: List[Verificacao] = Field(description="Verificações de conteúdo aplicadas se a resposta for 200 (pode ser lista vazia).")
    justificativa: str = Field(description="Por que o resultado esperado decorre da documentação.")
    evidencias: List[str] = Field(description="chunk_id dos trechos que sustentam o caso (obrigatório, ao menos um).")

class ConjuntoCasos(BaseModel):
    casos: List[CasoTeste]

In [ ]:
# ----------------------------- Prompts e cadeias (LCEL) -----------------------------
SISTEMA_BASE = ("Você é um engenheiro de QA que analisa documentação OpenAPI. Use SOMENTE as informações do contexto fornecido; "
                "não invente campos, parâmetros, códigos de resposta ou comportamentos que não estejam no texto.")

PROMPT_CLASSIFICACAO = ChatPromptTemplate.from_messages([
    ("system", SISTEMA_BASE),
    ("human",
     "Contexto recuperado da documentação (cada trecho começa com seu [chunk_id]):\n\n{contexto}\n\n"
     "TAREFA: CLASSIFICAÇÃO DE TEXTO. Com base apenas no texto acima, classifique a operação descrita quanto à "
     "intenção, ao risco de efeito colateral e à natureza do endpoint. Escreva uma justificativa curta em português e "
     "cite em 'evidencias' os chunk_id usados (sem os colchetes)."),
])

PROMPT_NER = ChatPromptTemplate.from_messages([
    ("system", SISTEMA_BASE),
    ("human",
     "Contexto recuperado da documentação (cada trecho começa com seu [chunk_id]):\n\n{contexto}\n\n"
     "TAREFA: RECONHECIMENTO DE ENTIDADES NOMEADAS. Extraia as entidades técnicas mencionadas no texto: recursos "
     "(ex.: Pet), identificadores, parâmetros, campos de schemas, restrições (obrigatoriedade, tipo, formato, enum), códigos de "
     "resposta HTTP e valores enumerados. Regras: (1) 'trecho_origem' deve ser cópia LITERAL de um trecho do contexto, no idioma original; "
     "(2) 'valor' deve aparecer dentro de 'trecho_origem'; (3) NÃO invente entidades ausentes; (4) não repita entidades; "
     "(5) retorne no máximo 40 entidades, priorizando as mais relevantes para testar a operação."),
])

PROMPT_GERACAO = ChatPromptTemplate.from_messages([
    ("system", SISTEMA_BASE),
    ("human",
     "OPERAÇÃO ALVO: {operacao}\n"
     "Códigos de resposta documentados: {codigos}\n"
     "Classificação textual da operação: {classificacao}\n"
     "Entidades extraídas: {entidades}\n\n"
     "Contexto recuperado da documentação (cada trecho começa com seu [chunk_id]):\n\n{contexto}\n\n"
     "TAREFA: GERAÇÃO DE CASOS DE TESTE com dados sintéticos. Gere de 4 a {n_max} casos (somente requisições GET) que verifiquem se o "
     "comportamento da API segue o contrato documentado. Regras:\n"
     "1. Use apenas parâmetros documentados, com o 'local' correto (path ou query) e valores sempre como string.\n"
     "2. Crie dados sintéticos VÁLIDOS e INVÁLIDOS justificados pelos schemas (tipo, formato, enum). Inclua no máximo um caso de limite "
     "(ex.: 0 ou -1). Valores inválidos devem estar claramente fora do contrato (texto onde se espera inteiro, valor fora do enum).\n"
     "3. 'codigos_esperados' deve conter SOMENTE códigos numéricos documentados. Não presuma o estado do servidor: um id que pode ou não "
     "existir deve aceitar todos os códigos documentados plausíveis (ex.: [200, 404]). A API é pública e os dados mudam.\n"
     "4. 'verificacoes' só valem para respostas 200; use APENAS o vocabulário fechado do schema. Não crie verificações que dependam de dados específicos.\n"
     "5. Se a operação não tem parâmetros, gere somente casos positivos (variações de verificação) e deixe 'parametros' vazio. Não invente parâmetros.\n"
     "6. Em 'evidencias' cite os chunk_id (sem colchetes) que sustentam cada caso; é obrigatório citar ao menos um."),
])

# Saída estruturada com include_raw=True: permite medir quantas saídas passam pelo schema
# (parsed=None indica falha de validação) sem que a falha interrompa o fluxo.
def montar_cadeia(prompt, schema):
    return (prompt | llm.with_structured_output(schema, include_raw=True)).with_retry(stop_after_attempt=3, wait_exponential_jitter=True)

CADEIA_CLASSIFICACAO = montar_cadeia(PROMPT_CLASSIFICACAO, ClassificacaoOperacao)
CADEIA_NER = montar_cadeia(PROMPT_NER, ExtracaoEntidades)
CADEIA_GERACAO = montar_cadeia(PROMPT_GERACAO, ConjuntoCasos)

ESTATISTICAS_LLM = []   # uma linha por tarefa/endpoint: a saída passou no schema na 1ª tentativa?

def chamar_estruturado(tarefa, op_id, cadeia, entradas, tentativas=3):
    """Invoca a cadeia. Falha de schema -> nova tentativa (só a 1ª conta na métrica). Erro de API -> registra e retorna None."""
    for t in range(1, tentativas + 1):
        parsed, falha = None, None
        try:
            saida = cadeia.invoke(entradas)
            parsed = saida.get("parsed")
            if parsed is None:
                falha = ("schema", str(saida.get("parsing_error"))[:200])
        except Exception as e:
            falha = ("api", repr(e)[:200])
        if t == 1:
            ESTATISTICAS_LLM.append({"tarefa": tarefa, "endpoint": op_id, "valido_1a_tentativa": parsed is not None,
                                     "tipo_falha": falha[0] if falha else None, "detalhe": falha[1] if falha else ""})
        time.sleep(PAUSA_ENTRE_CHAMADAS)
        if parsed is not None:
            return parsed
        if falha and falha[0] == "api":
            print(f"  ! Erro de API em {tarefa} / {op_id}: {falha[1]}")
            return None
    print(f"  ! {tarefa} / {op_id}: saída fora do schema após {tentativas} tentativas")
    return None

def carregar_cache(nome):
    arq = RESULTADOS / nome
    if USAR_CACHE_LLM and arq.exists():
        return json.loads(arq.read_text(encoding="utf-8"))
    return None

def salvar_json(nome, obj):
    (RESULTADOS / nome).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

### 5.1 Classificação de Textos

In [ ]:
CLASSIFICACOES, CONTEXTOS_CLS = {}, {}
_cache = carregar_cache("classificacao.json")

for op_id in REGISTROS:
    docs = recuperar(op_id, "Operation summary and description", k=4)
    CONTEXTOS_CLS[op_id] = docs
    if _cache and op_id in _cache:
        CLASSIFICACOES[op_id] = ClassificacaoOperacao.model_validate(_cache[op_id])
        continue
    r = chamar_estruturado("classificacao", op_id, CADEIA_CLASSIFICACAO, {"contexto": formatar_contexto(docs)})
    if r is not None:
        CLASSIFICACOES[op_id] = r

salvar_json("classificacao.json", {k: v.model_dump() for k, v in CLASSIFICACOES.items()})

# Referência objetiva: intenção esperada a partir do verbo HTTP (que o modelo NÃO viu)
INTENCAO_POR_METODO = {"GET": "consulta", "POST": "criacao", "PUT": "atualizacao", "PATCH": "atualizacao", "DELETE": "exclusao"}

df_classificacao = pd.DataFrame([{
    "endpoint": op, "intencao": c.intencao, "esperada (verbo HTTP)": INTENCAO_POR_METODO[REGISTROS[op]["metodo"]],
    "acertou": c.intencao == INTENCAO_POR_METODO[REGISTROS[op]["metodo"]],
    "risco": c.risco, "natureza": c.natureza, "justificativa": c.justificativa, "evidências": ", ".join(c.evidencias),
} for op, c in CLASSIFICACOES.items()])
df_classificacao

### 5.2 Reconhecimento de Entidades Nomeadas (NER)

In [ ]:
def normalizar(txt: str) -> str:
    return re.sub(r"\s+", " ", txt).strip().lower()

ENTIDADES, CONTEXTOS_NER = {}, {}
_cache = carregar_cache("entidades.json")

for op_id in REGISTROS:
    docs = recuperar(op_id, consulta_geral(op_id), k=10)
    CONTEXTOS_NER[op_id] = docs
    if _cache and op_id in _cache:
        ENTIDADES[op_id] = ExtracaoEntidades.model_validate(_cache[op_id])
        continue
    r = chamar_estruturado("ner", op_id, CADEIA_NER, {"contexto": formatar_contexto(docs)})
    if r is not None:
        ENTIDADES[op_id] = r

salvar_json("entidades.json", {k: v.model_dump() for k, v in ENTIDADES.items()})

# Checagem determinística anti-alucinação: o trecho citado existe literalmente na documentação recuperada?
linhas_ner = []
for op_id, ext in ENTIDADES.items():
    texto_ctx = normalizar(" ".join(d.page_content for d in CONTEXTOS_NER[op_id]))
    for e in ext.entidades:
        linhas_ner.append({"endpoint": op_id, "tipo": e.tipo, "valor": e.valor, "trecho_origem": e.trecho_origem[:90],
                           "fundamentada": normalizar(e.trecho_origem) in texto_ctx and normalizar(e.valor) in normalizar(e.trecho_origem)})
df_entidades = pd.DataFrame(linhas_ner)
print(f"Entidades extraídas: {len(df_entidades)} | fundamentadas na documentação: {int(df_entidades['fundamentada'].sum()) if len(df_entidades) else 0}")
if len(df_entidades):
    display(df_entidades.groupby(["endpoint", "tipo"]).size().unstack(fill_value=0))
df_entidades.head(25)

## **6. Geração dos casos de teste (dados sintéticos + rastreabilidade)**

O LLM recebe o contexto recuperado (RAG), a classificação e as entidades e devolve casos **estruturados**. Em seguida, **código determinístico valida cada caso** contra o contrato antes de qualquer requisição: parâmetros devem existir e ter o `local` correto, os códigos esperados devem estar entre os documentados e as evidências devem apontar para trechos realmente recuperados. Casos que violam as regras são **bloqueados** e não são executados.

In [ ]:
class CasoExecutavel(BaseModel):
    id: str
    endpoint: str
    origem: Literal["llm", "agente", "proposital"]
    caso: CasoTeste
    problemas: List[str] = []
    rastreavel: bool = False

def validar_caso(caso: CasoTeste, rec: dict) -> List[str]:
    """Regras determinísticas de validade de um caso gerado (antes de executar qualquer coisa)."""
    problemas = []
    documentados = {p["nome"]: p for p in rec["parametros"]}
    codigos_doc = {int(c) for c in rec["respostas"] if c.isdigit()}
    if not caso.nome.strip():
        problemas.append("caso sem nome")
    if not caso.codigos_esperados:
        problemas.append("sem códigos esperados")
    for c in caso.codigos_esperados:
        if c not in codigos_doc:
            problemas.append(f"código esperado {c} não está documentado")
    fornecidos = {}
    for p in caso.parametros:
        if p.nome not in documentados:
            problemas.append(f"parâmetro '{p.nome}' não documentado")
        elif documentados[p.nome]["local"] != p.local:
            problemas.append(f"parâmetro '{p.nome}' deveria estar em {documentados[p.nome]['local']}, não em {p.local}")
        fornecidos[p.nome] = p
    for nome, doc in documentados.items():
        if doc["obrigatorio"] and nome not in fornecidos:
            if doc["local"] == "path" or caso.tipo == "positivo":
                problemas.append(f"parâmetro obrigatório '{nome}' ausente")
    return problemas

def avaliar_rastreabilidade(caso: CasoTeste, ids_contexto: set) -> bool:
    ids = [e.strip("[] ") for e in caso.evidencias]
    return bool(ids) and all(i in ids_contexto for i in ids)

CONTADOR_CASOS = {"n": 0}

def gerar_casos(op_id: str, origem: str = "llm") -> List[CasoExecutavel]:
    """Gera, valida e empacota casos de teste de um endpoint (usa classificação e NER já calculados)."""
    rec = REGISTROS[op_id]
    docs = recuperar(op_id, consulta_geral(op_id), k=10)
    ids_ctx = {d.metadata["chunk_id"] for d in docs}
    entradas = {
        "operacao": op_id,
        "codigos": ", ".join(c for c in rec["respostas"] if c.isdigit()),
        "classificacao": CLASSIFICACOES[op_id].model_dump_json() if op_id in CLASSIFICACOES else "indisponível",
        "entidades": json.dumps([e.model_dump() for e in ENTIDADES[op_id].entidades][:40], ensure_ascii=False) if op_id in ENTIDADES else "indisponível",
        "contexto": formatar_contexto(docs), "n_max": MAX_CASOS_POR_OPERACAO,
    }
    conjunto = chamar_estruturado(f"geracao[{origem}]", op_id, CADEIA_GERACAO, entradas)
    saida = []
    for caso in (conjunto.casos[:MAX_CASOS_POR_OPERACAO] if conjunto else []):
        CONTADOR_CASOS["n"] += 1
        saida.append(CasoExecutavel(id=f"T{CONTADOR_CASOS['n']:02d}", endpoint=op_id, origem=origem, caso=caso,
                                    problemas=validar_caso(caso, rec), rastreavel=avaliar_rastreabilidade(caso, ids_ctx)))
    return saida

CASOS: List[CasoExecutavel] = []
_cache = carregar_cache("casos_gerados.json")
if _cache:
    CASOS = [CasoExecutavel.model_validate(c) for c in _cache if c["origem"] == "llm"]
    CONTADOR_CASOS["n"] = len(CASOS)
else:
    for op_id in REGISTROS:
        CASOS.extend(gerar_casos(op_id))

def resumo_param(ce):
    return ", ".join(f"{p.nome}={p.valor!r}" for p in ce.caso.parametros) or "—"

pd.DataFrame([{
    "id": ce.id, "endpoint": ce.endpoint, "tipo": ce.caso.tipo, "nome": ce.caso.nome, "parâmetros": resumo_param(ce),
    "esperado": ce.caso.codigos_esperados, "verificações": len(ce.caso.verificacoes),
    "rastreável": ce.rastreavel, "problemas": "; ".join(ce.problemas) or "—",
} for ce in CASOS])

## **7. Harness independente: execução controlada e julgamento determinístico**

O *harness* é **100% Python** e não consulta o LLM. Ele tem duas partes separadas:

* **Execução controlada:** só `GET`; só operações da lista permitida; só o *host* da Petstore; parâmetros documentados; valores codificados na URL (sem injeção); *timeout*; limite de tamanho da resposta; re-tentativas apenas para falhas transitórias.
* **Julgamento (função pura):** recebe o caso e a resposta e decide o resultado.

| Situação | Resultado |
|---|---|
| Falha de rede, *timeout*, resposta grande demais, HTTP 429/502/503/504 persistente | **inconclusivo** (não prova defeito da API) |
| HTTP 500 persistente e não documentado | **reprovado** |
| Código HTTP fora dos esperados | **reprovado** |
| Resposta 200 com corpo que não é JSON, que viola o *schema* do OpenAPI ou que falha em alguma verificação | **reprovado** |
| Caso tudo confere | **aprovado** |
| Caso inválido segundo o contrato ou regra de segurança | **bloqueado** (não é executado) |

In [ ]:
SESSAO = requests.Session()
SESSAO.headers.update({"Accept": "application/json", "User-Agent": "UFABC-PLN-api-test-harness/1.0"})
CODIGOS_TRANSITORIOS = {429, 500, 502, 503, 504}

def construir_requisicao(rec: dict, caso: CasoTeste):
    """Monta (url, query) aplicando as regras de segurança. Levanta ValueError se algo for inseguro."""
    if rec["metodo"] != "GET":
        raise ValueError("apenas GET é permitido")
    if (rec["metodo"], rec["caminho"]) not in OPERACOES_PERMITIDAS:
        raise ValueError("operação fora da lista permitida")
    caminho, query = rec["caminho"], {}
    for p in caso.parametros:
        v = p.valor
        if len(v) > 200 or re.search(r"[\x00-\x1f\x7f]", v):
            raise ValueError(f"valor do parâmetro '{p.nome}' rejeitado (tamanho ou caracteres de controle)")
        if p.local == "path":
            if not v.strip():
                raise ValueError(f"parâmetro de caminho '{p.nome}' vazio")
            if "{" + p.nome + "}" not in caminho:
                raise ValueError(f"'{p.nome}' não existe no caminho")
            caminho = caminho.replace("{" + p.nome + "}", quote(v, safe=""))
        else:
            query[p.nome] = v
    if "{" in caminho:
        raise ValueError("parâmetro de caminho sem valor")
    url = BASE_URL + caminho
    if urlparse(url).netloc != HOST_PERMITIDO:
        raise ValueError("host fora da lista permitida")
    return url, query

def executar_http(url: str, query: dict, cabecalhos: dict) -> dict:
    """Faz a requisição com timeout, limite de bytes e re-tentativas para falhas transitórias."""
    ultimo = {}
    for tentativa in range(1 + HTTP_TENTATIVAS_EXTRAS):
        t0 = time.perf_counter()
        try:
            r = SESSAO.get(url, params=query, headers=cabecalhos, timeout=HTTP_TIMEOUT, stream=True)
            corpo, excedeu = b"", False
            for pedaco in r.iter_content(65536):
                corpo += pedaco
                if len(corpo) > MAX_BYTES_RESPOSTA:
                    excedeu = True
                    break
            r.close()
            ultimo = {"status": r.status_code, "corpo": corpo, "content_type": r.headers.get("Content-Type", ""),
                      "ms": round((time.perf_counter() - t0) * 1000), "erro": None, "excedeu_limite": excedeu,
                      "tentativas": tentativa + 1, "url": r.url}
            if r.status_code not in CODIGOS_TRANSITORIOS or excedeu:
                return ultimo
        except requests.RequestException as e:
            ultimo = {"status": None, "corpo": b"", "content_type": "", "ms": round((time.perf_counter() - t0) * 1000),
                      "erro": type(e).__name__, "excedeu_limite": False, "tentativas": tentativa + 1, "url": url}
        time.sleep(1.5 * (tentativa + 1))
    return ultimo

# ----------------------------- Verificações de conteúdo (vocabulário fechado) -----------------------------
def aplicar_verificacao(v: Verificacao, corpo):
    """Retorna (ok, detalhe). Implementação determinística de cada verificação do vocabulário fechado."""
    if v.tipo == "json_raiz_tipo":
        alvo = {"array": list, "object": dict}.get(v.valores[0] if v.valores else "")
        if alvo is None:
            return False, f"valor de referência inválido: {v.valores}"
        return isinstance(corpo, alvo), f"raiz é {type(corpo).__name__}, esperado {v.valores[0]}"
    if v.tipo == "json_tem_chaves":
        if not isinstance(corpo, dict):
            return False, "raiz não é objeto"
        faltam = [k for k in v.valores if k not in corpo]
        return not faltam, f"chaves ausentes: {faltam}" if faltam else "todas as chaves presentes"
    if v.tipo == "todos_itens_campo_em":
        if not isinstance(corpo, list):
            return False, "raiz não é array"
        if not v.campo:
            return False, "verificação sem 'campo'"
        fora = [i for i, it in enumerate(corpo) if not isinstance(it, dict) or str(it.get(v.campo)) not in v.valores]
        return not fora, f"{len(fora)} item(ns) com '{v.campo}' fora de {v.valores}" if fora else f"{len(corpo)} item(ns) conferem"
    if v.tipo == "todos_valores_inteiros":
        if not isinstance(corpo, dict):
            return False, "raiz não é objeto"
        ruins = [k for k, x in corpo.items() if not isinstance(x, int) or isinstance(x, bool)]
        return not ruins, f"valores não inteiros em: {ruins[:5]}" if ruins else f"{len(corpo)} valores inteiros"
    if v.tipo == "campo_igual":
        if not isinstance(corpo, dict) or not v.campo or not v.valores:
            return False, "verificação mal formada ou raiz não é objeto"
        return str(corpo.get(v.campo)) == v.valores[0], f"{v.campo}={corpo.get(v.campo)!r}, esperado {v.valores[0]!r}"
    return False, f"verificação desconhecida: {v.tipo}"

def julgar(ce: CasoExecutavel, resposta: dict, schema_200) -> dict:
    """Função PURA de julgamento. Não usa LLM, rede nem estado global."""
    esperados = ce.caso.codigos_esperados
    if resposta.get("erro"):
        return {"resultado": "inconclusivo", "motivo": f"falha de rede/timeout ({resposta['erro']})", "verificacoes": []}
    if resposta.get("excedeu_limite"):
        return {"resultado": "inconclusivo", "motivo": "resposta excede o limite de tamanho do harness", "verificacoes": []}
    status = resposta["status"]
    if status not in esperados:
        if status == 500:
            return {"resultado": "reprovado", "motivo": "HTTP 500 persistente, não documentado para este caso", "verificacoes": []}
        if status in (429, 502, 503, 504):
            return {"resultado": "inconclusivo", "motivo": f"instabilidade do servidor (HTTP {status})", "verificacoes": []}
        return {"resultado": "reprovado", "motivo": f"HTTP {status} fora do esperado {esperados}", "verificacoes": []}
    verificacoes = []
    if status == 200:
        if "json" not in resposta.get("content_type", "").lower():
            verificacoes.append({"nome": "content_type_json", "ok": False, "detalhe": resposta.get("content_type", "")})
        try:
            corpo = json.loads(resposta["corpo"].decode("utf-8"))
        except Exception:
            return {"resultado": "reprovado", "motivo": "corpo da resposta 200 não é JSON válido", "verificacoes": verificacoes}
        if schema_200:
            erros = sorted(jsonschema.Draft7Validator(schema_200).iter_errors(corpo), key=lambda e: list(map(str, e.path)))[:3]
            if erros:
                for e in erros:
                    verificacoes.append({"nome": "schema_openapi_200", "ok": False,
                                         "detalhe": f"/{'/'.join(map(str, e.path))}: {e.message[:140]}"})
            else:
                verificacoes.append({"nome": "schema_openapi_200", "ok": True, "detalhe": "corpo conforme o schema do OpenAPI"})
        for v in ce.caso.verificacoes:
            ok, detalhe = aplicar_verificacao(v, corpo)
            verificacoes.append({"nome": f"{v.tipo}({v.campo or ''})", "ok": ok, "detalhe": detalhe})
    falhas = [v for v in verificacoes if not v["ok"]]
    if falhas:
        return {"resultado": "reprovado", "motivo": "; ".join(f"{f['nome']}: {f['detalhe']}" for f in falhas)[:300], "verificacoes": verificacoes}
    return {"resultado": "aprovado", "motivo": f"HTTP {status} esperado" + (" e verificações ok" if verificacoes else ""), "verificacoes": verificacoes}

def executar_caso(ce: CasoExecutavel) -> dict:
    rec = REGISTROS[ce.endpoint]
    base = {"id": ce.id, "endpoint": ce.endpoint, "origem": ce.origem, "tipo": ce.caso.tipo, "nome": ce.caso.nome,
            "parametros": resumo_param(ce), "esperado": ce.caso.codigos_esperados, "evidencias": ", ".join(ce.caso.evidencias),
            "http": None, "ms": None, "tentativas": 0}
    if ce.problemas:
        return {**base, "resultado": "bloqueado", "motivo": "caso inválido: " + "; ".join(ce.problemas)}
    try:
        url, query = construir_requisicao(rec, ce.caso)
    except ValueError as e:
        return {**base, "resultado": "bloqueado", "motivo": f"regra de segurança: {e}"}
    cabecalhos = {"api_key": PETSTORE_API_KEY} if "api_key" in rec["seguranca"] else {}
    resposta = executar_http(url, query, cabecalhos)
    schema_200 = rec["respostas"].get("200", {}).get("schema_json")
    veredito = julgar(ce, resposta, schema_200)
    time.sleep(0.3)
    return {**base, "http": resposta.get("status"), "ms": resposta.get("ms"), "tentativas": resposta.get("tentativas"),
            "resultado": veredito["resultado"], "motivo": veredito["motivo"]}

### 7.1 Autoteste do julgador (sem rede e sem LLM)

Antes de executar contra a Petstore, provamos que o julgador distingue os três resultados usando respostas **simuladas**. Isso mostra que o veredito depende só do código Python.

In [ ]:
def _caso_falso(esperados, verificacoes=()):
    return CasoExecutavel(id="X", endpoint="GET /store/inventory", origem="proposital", rastreavel=True,
                          caso=CasoTeste(nome="autoteste", objetivo="", tipo="positivo", parametros=[], codigos_esperados=esperados,
                                         verificacoes=list(verificacoes), justificativa="", evidencias=["x"]))

_schema = {"type": "object", "additionalProperties": {"type": "integer"}}
_ok_json = {"status": 200, "corpo": b'{"sold": 3, "available": 5}', "content_type": "application/json", "erro": None, "excedeu_limite": False}
_v_chaves = Verificacao(tipo="json_tem_chaves", campo=None, valores=["sold"])
_v_tipo_errado = Verificacao(tipo="json_raiz_tipo", campo=None, valores=["array"])

_autotestes = [
    ("200 válido, schema e verificação ok",  _caso_falso([200], [_v_chaves]),       _ok_json, "aprovado"),
    ("404 quando só 200 era esperado",        _caso_falso([200]),                    {**_ok_json, "status": 404}, "reprovado"),
    ("200 com corpo violando o schema",       _caso_falso([200]),                    {**_ok_json, "corpo": b'{"sold": "muitos"}'}, "reprovado"),
    ("200 com verificação de conteúdo falha", _caso_falso([200], [_v_tipo_errado]),  _ok_json, "reprovado"),
    ("timeout de rede",                       _caso_falso([200]),                    {"status": None, "erro": "Timeout", "excedeu_limite": False}, "inconclusivo"),
    ("HTTP 503 do servidor",                  _caso_falso([200]),                    {**_ok_json, "status": 503}, "inconclusivo"),
]
_linhas = []
for desc, caso_f, resp_f, esperado_res in _autotestes:
    obtido = julgar(caso_f, resp_f, _schema)["resultado"]
    _linhas.append({"situação simulada": desc, "esperado": esperado_res, "julgador": obtido, "ok": obtido == esperado_res})
df_autoteste = pd.DataFrame(_linhas)
assert df_autoteste["ok"].all(), "O julgador não se comportou como especificado"
print(f"Autoteste do julgador: {int(df_autoteste['ok'].sum())}/{len(df_autoteste)} situações corretas")
df_autoteste

### 7.2 Teste inválido proposital

Para demonstrar que o harness **identifica reprovação** sem acusar defeito inexistente da Petstore, inserimos de propósito um caso cuja expectativa contradiz o contrato: o OpenAPI diz que `GET /pet/findByStatus` retorna um **array**, mas o caso afirma que a raiz deve ser um **objeto**. O esperado é que ele seja **reprovado**, e isso conta como acerto do harness.

In [ ]:
CASO_PROPOSITAL = CasoExecutavel(
    id="P01", endpoint="GET /pet/findByStatus", origem="proposital", rastreavel=True,
    caso=CasoTeste(
        nome="[proposital] raiz deveria ser objeto", objetivo="Expectativa deliberadamente contrária ao contrato (o contrato diz array).",
        tipo="positivo", parametros=[Parametro(nome="status", local="query", valor="available")], codigos_esperados=[200],
        verificacoes=[Verificacao(tipo="json_raiz_tipo", campo=None, valores=["object"])],
        justificativa="Teste inválido inserido de propósito para provar que o harness detecta reprovação.",
        evidencias=["GET /pet/findByStatus#resp-200"]))

## **8. Execução controlada contra a Swagger Petstore**

In [ ]:
def executar_lote(casos: List[CasoExecutavel]) -> pd.DataFrame:
    linhas = []
    for ce in casos:
        linha = executar_caso(ce)
        linhas.append(linha)
        print(f"{ce.id:>4} | {ce.endpoint:<20} | {linha['resultado']:<12} | {linha['motivo'][:80]}")
    return pd.DataFrame(linhas)

print("Executando casos gerados pelo LLM + caso proposital...\n")
df_exec = executar_lote(CASOS + [CASO_PROPOSITAL])
salvar_json("casos_gerados.json", [c.model_dump() for c in CASOS + [CASO_PROPOSITAL]])
df_exec

## **9. Agente com ferramentas controladas (extra)**

Um **agente** LangChain recebe um pedido em linguagem natural e decide **quais ferramentas usar**: listar operações, consultar a documentação (RAG) e solicitar a geração de testes. Nenhuma ferramenta executa requisições HTTP: o agente só planeja e propõe, e a execução continua sob controle do harness. Os casos do agente são marcados com `origem = "agente"`.

In [ ]:
CASOS_AGENTE: List[CasoExecutavel] = []
LOG_FERRAMENTAS: List[str] = []

@tool
def listar_operacoes() -> str:
    """Lista as operações da API liberadas para teste (método e caminho)."""
    LOG_FERRAMENTAS.append("listar_operacoes()")
    return json.dumps(list(REGISTROS.keys()), ensure_ascii=False)

@tool
def consultar_documentacao(endpoint: str) -> str:
    """Recupera (RAG) os trechos da documentação OpenAPI de um endpoint. Exemplo de argumento: 'GET /store/inventory'."""
    LOG_FERRAMENTAS.append(f"consultar_documentacao({endpoint!r})")
    if endpoint not in REGISTROS:
        return f"Endpoint não permitido. Use um destes: {list(REGISTROS.keys())}"
    return formatar_contexto(recuperar(endpoint, consulta_geral(endpoint), k=10))

@tool
def solicitar_geracao_de_testes(endpoint: str) -> str:
    """Pede a geração de casos de teste para um endpoint permitido. NÃO executa requisições HTTP: apenas registra os casos propostos."""
    LOG_FERRAMENTAS.append(f"solicitar_geracao_de_testes({endpoint!r})")
    if endpoint not in REGISTROS:
        return f"Endpoint não permitido. Use um destes: {list(REGISTROS.keys())}"
    novos = gerar_casos(endpoint, origem="agente")
    CASOS_AGENTE.extend(novos)
    validos = sum(1 for c in novos if not c.problemas)
    return f"{len(novos)} casos propostos para {endpoint} ({validos} válidos pelas regras do contrato). Execução será feita pelo harness."

try:
    try:
        from langchain.agents import create_agent
        def criar_agente(modelo, ferramentas, instrucoes):
            return create_agent(model=modelo, tools=ferramentas, system_prompt=instrucoes)
    except ImportError:  # versões antigas do LangChain
        from langgraph.prebuilt import create_react_agent
        def criar_agente(modelo, ferramentas, instrucoes):
            return create_react_agent(modelo, ferramentas, prompt=instrucoes)

    # Como subagente, o Gemini 3.5 Flash-Lite funciona melhor com raciocínio médio (o padrão é mínimo).
    try:
        llm_agente = ChatGoogleGenerativeAI(model=MODEL_ID, max_retries=2, timeout=90, thinking_level="medium")
    except Exception:
        llm_agente = llm

    agente = criar_agente(
        llm_agente, [listar_operacoes, consultar_documentacao, solicitar_geracao_de_testes],
        "Você é um agente de QA. Você NÃO executa requisições HTTP. Seu trabalho é: consultar a documentação do endpoint pedido "
        "e solicitar a geração de testes por meio das ferramentas. Ao final, resuma em português o que foi feito.")
    pedido = "Prepare testes para a operação GET /store/inventory: consulte a documentação e solicite a geração dos casos."
    resultado_agente = agente.invoke({"messages": [{"role": "user", "content": pedido}]})

    print("Ferramentas chamadas pelo agente, em ordem:")
    for i, f in enumerate(LOG_FERRAMENTAS, 1):
        print(f"  {i}. {f}")
    print("\nResposta final do agente:\n", texto_da_resposta(resultado_agente["messages"][-1]).strip())
except Exception as e:
    print(f"Não foi possível executar o agente nesta sessão: {e!r}")

print(f"\nCasos propostos pelo agente: {len(CASOS_AGENTE)}")

In [ ]:
if CASOS_AGENTE:
    print("Executando os casos propostos pelo agente (execução e julgamento pelo harness)...\n")
    df_agente = executar_lote(CASOS_AGENTE)
    df_exec = pd.concat([df_exec, df_agente], ignore_index=True)
    salvar_json("casos_gerados.json", [c.model_dump() for c in CASOS + [CASO_PROPOSITAL] + CASOS_AGENTE])
    df_agente
else:
    print("Sem casos do agente para executar.")

## **10. Resultados, métricas e figuras**

O caso **proposital** é contabilizado à parte: ele mede o harness, não a API. As demais métricas consideram apenas casos gerados pelo LLM (diretamente ou via agente).

In [ ]:
df_exec.to_csv(RESULTADOS / "relatorio_execucao.csv", index=False, encoding="utf-8")

df_api = df_exec[df_exec["origem"] != "proposital"]
executados = df_api[df_api["resultado"].isin(["aprovado", "reprovado", "inconclusivo"])]
todos_gerados = CASOS + CASOS_AGENTE

estat = pd.DataFrame(ESTATISTICAS_LLM)
validos_schema = int(estat["valido_1a_tentativa"].sum()) if len(estat) else 0
saidas_geradas = int((estat["tipo_falha"] != "api").sum()) if len(estat) else 0   # exclui falhas de API (cota, rede)

metricas = {
    "casos_gerados": len(todos_gerados),
    "casos_executados": len(executados),
    "casos_bloqueados": int((df_api["resultado"] == "bloqueado").sum()),
    "aprovados": int((executados["resultado"] == "aprovado").sum()),
    "reprovados": int((executados["resultado"] == "reprovado").sum()),
    "inconclusivos": int((executados["resultado"] == "inconclusivo").sum()),
    "cobertura_de_operacoes_pct": round(100 * executados["endpoint"].nunique() / len(REGISTROS), 1),
    "validade_estrutural_pct": round(100 * validos_schema / saidas_geradas, 1) if saidas_geradas else 0.0,
    "rastreabilidade_pct": round(100 * sum(c.rastreavel for c in todos_gerados) / len(todos_gerados), 1) if todos_gerados else 0.0,
    "classificacao_correta_pct": round(100 * df_classificacao["acertou"].mean(), 1) if len(df_classificacao) else 0.0,
    "entidades_fundamentadas_pct": round(100 * df_entidades["fundamentada"].mean(), 1) if len(df_entidades) else 0.0,
    "harness_detectou_caso_proposital": bool((df_exec[df_exec["origem"] == "proposital"]["resultado"] == "reprovado").all()),
}
salvar_json("metricas.json", metricas)
pd.DataFrame({"métrica": list(metricas.keys()), "valor": list(metricas.values())})

In [ ]:
# Distribuição de resultados por endpoint (casos gerados pelo LLM/agente) e métricas percentuais
fig, eixos = plt.subplots(1, 2, figsize=(14, 4.5))

if len(df_api):
    tabela = df_api.pivot_table(index="endpoint", columns="resultado", values="id", aggfunc="count", fill_value=0)
    cores = {"aprovado": "#2e8b57", "reprovado": "#c0392b", "inconclusivo": "#e0a800", "bloqueado": "#7f8c8d"}
    tabela.plot(kind="barh", stacked=True, ax=eixos[0], color=[cores.get(c, "#999") for c in tabela.columns])
eixos[0].set_title("Resultado dos casos por endpoint")
eixos[0].set_xlabel("número de casos"); eixos[0].set_ylabel("")

nomes = ["Cobertura de\noperações", "Validade\nestrutural", "Rastreabilidade", "Classificação\ncorreta", "Entidades\nfundamentadas"]
valores = [metricas["cobertura_de_operacoes_pct"], metricas["validade_estrutural_pct"], metricas["rastreabilidade_pct"],
           metricas["classificacao_correta_pct"], metricas["entidades_fundamentadas_pct"]]
barras = eixos[1].bar(nomes, valores, color="#2c6fbb")
eixos[1].set_ylim(0, 110); eixos[1].set_ylabel("%"); eixos[1].set_title("Métricas de qualidade do pipeline")
for b, v in zip(barras, valores):
    eixos[1].text(b.get_x() + b.get_width() / 2, v + 2, f"{v:.0f}%", ha="center")
plt.tight_layout()
plt.savefig(RESULTADOS / "figura_resultados.png", dpi=150)
plt.show()

In [ ]:
# Relatório detalhado de casos reprovados/inconclusivos/bloqueados (com motivo e evidência documental)
problematicos = df_exec[df_exec["resultado"].isin(["reprovado", "inconclusivo", "bloqueado"])]
problematicos[["id", "endpoint", "origem", "nome", "parametros", "esperado", "http", "resultado", "motivo", "evidencias"]]

## **11. Verificação dos critérios de conclusão e exportação**

In [ ]:
checklist = {
    "Classificação de Textos gerou saídas observáveis": len(CLASSIFICACOES) > 0,
    "Reconhecimento de Entidades Nomeadas gerou saídas observáveis": len(df_entidades) > 0,
    "Ao menos um conjunto de testes foi executado": metricas["casos_executados"] > 0,
    "Julgamento feito por código Python (autoteste do julgador ok)": bool(df_autoteste["ok"].all()),
    "Harness reprovou o caso proposital": metricas["harness_detectou_caso_proposital"],
    "Cada caso gerado tem endpoint e evidência documental": metricas["rastreabilidade_pct"] > 0,
}
for item, ok in checklist.items():
    print("✔" if ok else "✘", item)

# Empacota os resultados (sem segredos) para anexar ao repositório do GitHub
shutil.make_archive("resultados_pln", "zip", RESULTADOS)
print("\nArquivos em results/:", ", ".join(sorted(p.name for p in RESULTADOS.iterdir())))
try:
    from google.colab import files
    files.download("resultados_pln.zip")
except Exception:
    print("Download automático indisponível fora do Colab; use o arquivo resultados_pln.zip.")

## **12. Conclusões e limitações**

**O que o experimento mostra**

* O LLM foi usado como **interpretador de texto** (classificação e NER) e como **gerador de cenários**; a decisão de aprovado/reprovado ficou inteiramente com o harness, o que torna a avaliação objetiva e repetível: o mesmo caso, com a mesma resposta, sempre recebe o mesmo veredito (comprovado pelo autoteste do julgador).
* A **rastreabilidade** liga cada teste ao endpoint e ao trecho de documentação (`chunk_id`) que o sustenta, e as checagens determinísticas medem **alucinação** (entidades sem fundamento no texto, parâmetros ou códigos inexistentes no contrato).
* Casos **bloqueados** mostram o valor de validar a saída do LLM contra o contrato *antes* de executar qualquer coisa.
* **Reprovado** significa que o comportamento observado divergiu do contrato documentado. Como a Petstore é uma API pública de demonstração, a divergência pode estar na documentação, na implementação ou nos dados compartilhados, e deve ser analisada caso a caso nos motivos do relatório. Falhas de rede e instabilidade ficam como **inconclusivo**, nunca como defeito da API.

**Limitações**

* Escopo restrito a três operações `GET`; sem testes de carga, autenticação avançada, operações destrutivas ou mutações concorrentes.
* A Petstore é compartilhada: dados e respostas variam entre execuções; por isso os casos aceitam todos os códigos documentados plausíveis quando o estado do servidor é desconhecido.
* O vocabulário de verificações é fechado (5 tipos), o que garante julgamento determinístico, mas limita a expressividade dos testes.
* Modelos de linguagem não são determinísticos: os resultados do LLM podem variar entre execuções; os JSONs em `results/` registram a execução apresentada.
* Disponibilidade, cotas e identificador do modelo devem ser conferidos no momento da execução.